# 2.文件对话消息历史组件实现记忆

知识点讲解：FileChatMessageHistory 文件持久化记忆

1. FileChatMessageHistory 的定位
   - 位于 langchain_community.chat_message_histories，是 BaseChatMessageHistory 的文件系统实现
   - 把消息序列化成 JSON 存到本地文件，最大优势是「进程重启后记忆仍在」
   - 定位：单机、单用户、轻量级持久化，是理解持久化机制的最佳入门实现

2. 底层工作原理
   - 构造时接收一个文件路径，若文件不存在会自动创建
   - messages 属性：每次读取都从磁盘加载并反序列化 JSON → List[BaseMessage]
   - add_message()：先读全量、追加、再全量写回（覆盖式写入）
   - 序列化依赖 langchain_core.messages 的 messages_to_dict / messages_from_dict

3. 性能特征与适用边界
   - 每次读写都是全量 IO，消息越多越慢，时间复杂度 O(n)
   - 无文件锁，多进程并发写入会互相覆盖，存在数据丢失风险
   - 适用：本地 CLI 工具、个人助手、demo 演示
   - 不适用：Web 服务、多用户、高并发场景（应换 Redis / 数据库实现）

4. 本示例的「存储 + 处理」组合（课程标准写法）
   - 记忆的存储：FileChatMessageHistory("./memory.txt") 负责把对话落盘
   - 记忆的处理逻辑：在每次循环里构造 system_prompt，用 f-string 把历史注入 <context> 标签：
       system_prompt = "...
<context>
{chat_history}
</context>
"
   - f"{chat_history}" 会调用对象的 __str__，得到消息的文本表示
   - 这是「原生 OpenAI SDK + LangChain 存储组件」混用的过渡写法；纯 LangChain 方案应使用 MessagesPlaceholder + RunnableWithMessageHistory

5. 记忆写入的时机（非常关键）
   - 必须在流式响应「完全消费完毕」之后才调用 add_user_message / add_ai_message
   - 原因：流式返回是增量的，中途 ai_content 还不完整
   - 课程标准写法在循环结束后统一调用 add_messages([HumanMessage(query), AIMessage(ai_content)])
   - 顺序也有讲究：先 add_user_message 再 add_ai_message，保证消息顺序正确

6. 推理模型的流式处理要点
   - extra_body={"thinking": {"type": "disabled"}} 关闭深度思考，降低延迟与成本
   - 即便关闭思考，仍保留 reasoning_content 的兼容处理，属于防御性编程
   - 判断条件必须用 if delta.content: 而非 content is None: break，否则思考阶段的第一个 chunk 就会中断整个流

7. 与其他持久化实现的切换
   - FileChatMessageHistory("./memory.txt")
   - RedisChatMessageHistory(session_id="u1", url="redis://...", ttl=3600)
   - SQLChatMessageHistory(session_id="u1", connection_string="sqlite:///chat.db")
   - 三者 API 完全一致，切换存储后端只需改一行构造代码

8. 注意事项
   - 文件名虽为 .txt，内容实际是 JSON 格式
   - 历史会无限增长，长期运行需配合 trim_messages 或定期归档
   - 文件中会明文保存用户对话，涉及隐私数据时需加密或脱敏
   - 多用户场景下应按 user_id 拆分文件路径，避免记忆串号


In [ ]:
import os
import dotenv
from langchain_community.chat_message_histories import FileChatMessageHistory
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam
dotenv.load_dotenv()
client = OpenAI(base_url=os.getenv("OPENAI_API_BASE"))
chat_history = FileChatMessageHistory("./memory.txt")
while True:
    query = input("Human: ")
    if query == "q":
        exit(0)
    print("AI: ", flush=True, end="")
    system_prompt = (
        "你是DeepSeek开发的DeepSeek聊天机器人，可以根据相应的上下文回复用户信息，上下文里存放的是人类与你对话的信息列表。\n\n"
        f"<context>{chat_history}</context>\n\n"
    )
    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]
    response = client.chat.completions.create(
        model='deepseek-v4-pro',
        extra_body={"thinking": {"type": "disabled"}},
        messages=messages,
        stream=True,
    )
    ai_content = ""
    in_reasoning = False
    for chunk in response:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        reasoning = getattr(delta, "reasoning_content", None)
        if reasoning:
            if not in_reasoning:
                print("[思考] ", flush=True, end="")
                in_reasoning = True
            print(reasoning, flush=True, end="")
        if delta.content:
            if in_reasoning:
                print("\n[回答] ", flush=True, end="")
                in_reasoning = False
            ai_content += delta.content
            print(delta.content, flush=True, end="")
    chat_history.add_user_message(query)
    chat_history.add_ai_message(ai_content)
    print("")